# Signal vs Market Baseline · IHSG

## 1 · Tujuan dan pertanyaan
Apakah aturan regime SMA 50/200 dengan jeda satu sesi menghasilkan return dan drawdown berbeda dari IHSG buy-and-hold pada tanggal identik? Ini eksperimen deskriptif historis dengan aturan tetap, belum bukti alpha teruji pada holdout baru. IHSG digunakan sebagai proksi riset. Notebook ini tidak mengirim order atau mengambil API. Semua output berikut dihitung dari berkas Sectors lokal yang dilampirkan.

## 2 · Imports dan konfigurasi
Python standard library menjalankan replay dan bootstrap. Pandas dan matplotlib digunakan untuk tabel dan grafik. Seed 42, 500 bootstrap blok 20 sesi; parameter tetap sebelum melihat hasil.

In [ ]:
import sys, os, json, math, hashlib, platform
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
print('Python:', platform.python_version(), 'pandas:', pd.__version__)
SEED = 42

## 3 · Runtime dan sumber
Lampirkan `market-overview-data.json` dari `docs/prototypes/`. Colab: unggah file lewat Files. Kaggle: tambahkan file tersebut sebagai Dataset input. Lokal: jalankan dari root workspace atau isi variabel lingkungan `IDXEL_MARKET_BUNDLE`. Tidak ada fallback internet.

In [ ]:
candidates = [Path(os.environ.get('IDXEL_MARKET_BUNDLE', 'market-overview-data.json')), Path('docs/prototypes/market-overview-data.json'), Path('../docs/prototypes/market-overview-data.json'), Path('/content/market-overview-data.json')]
if Path('/kaggle/input').exists():
    candidates += list(Path('/kaggle/input').rglob('market-overview-data.json'))
INPUT = next((p for p in candidates if p.is_file()), None)
if INPUT is None:
    raise FileNotFoundError('Lampirkan market-overview-data.json dari docs/prototypes/ atau set IDXEL_MARKET_BUNDLE.')
print('Input:', INPUT)

## 4 · Loading dan inspeksi sumber
Bundle lokal memuat harga IHSG dan metadata provenance dari snapshot Sectors. Universe saham tidak dipakai agar eksperimen ini tidak bergantung pada anggota LQ45 masa kini.

In [ ]:
bundle = json.loads(INPUT.read_text())
ihsg = bundle['ihsg']
prices = pd.DataFrame(ihsg['series'])[['date','price']]
print('Provider:', ihsg['provider'], 'status:', ihsg['data_quality'])
print('Snapshot:', ihsg['source_files'])
print('Coverage:', ihsg['coverage_start'], '—', ihsg['coverage_end'])
display(prices.head()); display(prices.tail())

## 5 · Audit data
Tolak sumber yang belum VERIFIED, tanggal duplikat/tidak urut, atau harga tidak positif/tidak finite. Price index tidak menjelaskan biaya ETF/fund, dividen, dan fill transaksi aktual.

In [ ]:
audit = pd.DataFrame([{'rows': len(prices), 'duplicate_dates': int(prices.date.duplicated().sum()), 'missing_prices': int(prices.price.isna().sum()), 'ordered': prices.date.is_monotonic_increasing, 'positive': bool((prices.price>0).all()), 'status': ihsg['data_quality']}])
display(audit)
assert ihsg['ok'] and ihsg['data_quality']=='VERIFIED'
assert prices.date.is_unique and prices.date.is_monotonic_increasing
assert prices.price.map(math.isfinite).all() and (prices.price>0).all()

## 6 · EDA
Grafik harga dan distribusi return memberi konteks penurunan/kenaikan pasar. Distribusi teramati bukan probabilitas return ke depan.

In [ ]:
daily = prices.price.pct_change().dropna()
fig, ax = plt.subplots(1,2, figsize=(12,3))
ax[0].plot(pd.to_datetime(prices.date), prices.price); ax[0].set(title='IHSG · price index', ylabel='Poin indeks')
ax[1].hist(daily*100, bins=35); ax[1].set(title='Return close-to-close', xlabel='Return harian (%)', ylabel='Sesi')
for a in ax: a.grid(alpha=.2)
plt.tight_layout(); plt.show()

## 7 · Feature engineering dan availability
Syarat eksposur: close > SMA200 dan SMA50 > SMA200. Observasi close pada sesi s baru menentukan posisi pada close s+1, lalu menghasilkan return s+1→s+2. Shift dua bar mencegah asumsi fill pada close yang sama dengan observasi.

In [ ]:
features=prices.copy()
features['sma50']=features.price.rolling(50).mean()
features['sma200']=features.price.rolling(200).mean()
features['signal']=(features.price>features.sma200)&(features.sma50>features.sma200)
features['position_for_return']=features.signal.shift(2)
display(features.iloc[195:205])

## 8 · Pemanasan dan batas evaluasi
200 penutupan pertama adalah pemanasan; bar index 200 menjadi titik awal wealth=1. Return uji pertama adalah index 201. Model tidak fitted sehingga tidak ada normalisasi/train-test fitting. Ini replay rolling dengan parameter tetap, bukan untouched holdout.

In [ ]:
assert len(prices)>=301
print('Pemanasan:', prices.date.iloc[0], '—', prices.date.iloc[199])
print('Awal wealth:', prices.date.iloc[200])
print('Return uji:', prices.date.iloc[201], '—', prices.date.iloc[-1])
print('Jumlah return:', len(prices)-201)

## 9 · Baseline
IHSG buy-and-hold dimulai dari close yang sama, dengan biaya masuk satu arah yang sama. Return kumulatif adalah wealth akhir−1; selisih adalah percentage points, bukan return relatif signal/market.

In [ ]:
benchmark_gross=prices.price.iloc[-1]/prices.price.iloc[200]-1
print('IHSG kumulatif bruto:', f'{benchmark_gross:.2%}')

## 10 · Definisi model/replay lengkap
Berikut kode lengkap yang sama dengan implementasi web. Posisi biner 0/1, kas 0% yield, biaya pada setiap perubahan eksposur. Wealth initial 1 ikut dalam perhitungan puncak drawdown. Bootstrap meresample blok pasangan return dari lintasan yang sudah teramati; ia tidak menjalankan ulang sinyal pada harga sintetik.

In [ ]:
"""Fixed, causal IHSG regime replay against the identical IHSG market path.

No forecasting, tuning, constituent selection or external calls. The index is
a research proxy, not a tradable instrument. Execution uses a full session
delay because the input contains closes only, not verified open/VWAP fills.
"""

from __future__ import annotations

from datetime import date
import hashlib
import json
import math
import random
from statistics import mean, stdev
from typing import Any


VERSION = "ihsg-regime-baseline.v1"


def _quantile(values: list[float], probability: float) -> float:
    ordered = sorted(values)
    index = (len(ordered) - 1) * probability
    lower = int(index)
    upper = min(lower + 1, len(ordered) - 1)
    return ordered[lower] + (ordered[upper] - ordered[lower]) * (index - lower)


def paired_block_interval(pairs: list[tuple[float, float]], *, seed: int = 42,
                          simulations: int = 500, block: int = 20) -> dict[str, Any]:
    """95% interval for annualized arithmetic excess, resampling paired blocks.

    Conditional on this realized fixed-rule path. Does not refit/replay signals
    on synthetic prices and does not estimate future strategy performance.
    """
    if len(pairs) < block * 5:
        return {"status": "INSUFFICIENT_SAMPLE", "lower": None, "upper": None}
    rng = random.Random(seed)
    differences = [strategy - market for strategy, market in pairs]
    sampled_means = []
    for _ in range(simulations):
        sample = []
        while len(sample) < len(pairs):
            start = rng.randrange(len(pairs) - block + 1)
            sample.extend(differences[start:start + block])
        sampled_means.append(mean(sample[:len(pairs)]) * 252)
    return {"status": "EXPLORATORY", "estimate": mean(differences) * 252,
            "lower": _quantile(sampled_means, .025), "upper": _quantile(sampled_means, .975),
            "seed": seed, "simulations": simulations, "block_sessions": block,
            "sample_count": len(pairs), "metric": "annualized_arithmetic_excess_return",
            "interpretation": "Conditional paired-block interval on the observed path, not a forecast or proof of alpha."}


def _metrics(returns: list[float], wealth: float, mdd: float) -> dict[str, float]:
    return {"cumulative_return": wealth - 1,
            "annualized_return": wealth ** (252 / len(returns)) - 1,
            "annualized_volatility": stdev(returns) * math.sqrt(252) if len(returns) > 1 else 0,
            "max_drawdown": mdd}


def _replay(series: list[dict[str, Any]], cost_bps: float) -> dict[str, Any]:
    prices = [row["price"] for row in series]
    # First rule observation is close index 199 (200 completed sessions).
    # Observe at s, establish the hypothetical position at close s+1,
    # earn close s+1 -> close s+2. Never fill at the observation close.
    start = 200
    path = [{"date": series[start]["date"], "signal_wealth": 1.0,
             "market_wealth": 1.0, "position": None, "signal_date": None,
             "signal_drawdown": 0.0, "market_drawdown": 0.0}]
    strategy_wealth = market_wealth = signal_peak = market_peak = 1.0
    signal_mdd = market_mdd = 0.0
    previous_position = 0
    switches = 0
    paired_returns: list[tuple[float, float]] = []
    signal_returns, market_returns, rows = [], [], []
    fee = cost_bps / 10_000
    prefix = [0.0]
    for value in prices:
        prefix.append(prefix[-1] + value)
    for t in range(start + 1, len(series)):
        signal_index = t - 2
        sma50 = (prefix[signal_index + 1] - prefix[signal_index - 49]) / 50
        sma200 = (prefix[signal_index + 1] - prefix[signal_index - 199]) / 200
        position = int(prices[signal_index] > sma200 and sma50 > sma200)
        turnover = abs(position - previous_position)
        switches += turnover
        market_return = prices[t] / prices[t - 1] - 1
        # Cost is paid on each 0/1 exposure change, including first entry.
        # Benchmark pays the same one-way entry cost, once, at evaluation start.
        signal_net = (1 + position * market_return) * (1 - fee * turnover) - 1
        market_net = (1 + market_return) * (1 - fee * int(t == start + 1)) - 1
        strategy_wealth *= 1 + signal_net
        market_wealth *= 1 + market_net
        signal_peak = max(signal_peak, strategy_wealth)
        market_peak = max(market_peak, market_wealth)
        signal_dd, market_dd = strategy_wealth / signal_peak - 1, market_wealth / market_peak - 1
        signal_mdd, market_mdd = min(signal_mdd, signal_dd), min(market_mdd, market_dd)
        row = {"date": series[t]["date"], "signal_date": series[signal_index]["date"],
               "position_date": series[t - 1]["date"], "position": position,
               "turnover": turnover, "signal_return": signal_net, "market_return": market_net,
               "signal_wealth": strategy_wealth, "market_wealth": market_wealth,
               "signal_drawdown": signal_dd, "market_drawdown": market_dd}
        rows.append(row)
        path.append(row)
        signal_returns.append(signal_net)
        market_returns.append(market_net)
        paired_returns.append((signal_net, market_net))
        previous_position = position
    quarterly = []
    quarters = sorted({f"{row['date'][:4]}-Q{(int(row['date'][5:7]) - 1) // 3 + 1}" for row in rows})
    for quarter in quarters:
        subset = [row for row in rows if f"{row['date'][:4]}-Q{(int(row['date'][5:7]) - 1) // 3 + 1}" == quarter]
        sr = math.prod(1 + row["signal_return"] for row in subset) - 1
        mr = math.prod(1 + row["market_return"] for row in subset) - 1
        quarterly.append({"period": quarter, "start": subset[0]["date"], "end": subset[-1]["date"],
                          "sessions": len(subset), "signal_return": sr, "market_return": mr,
                          "excess": sr - mr})
    return {"cost_bps": cost_bps, "path": path,
            "signal": _metrics(signal_returns, strategy_wealth, signal_mdd),
            "market": _metrics(market_returns, market_wealth, market_mdd),
            "excess_cumulative": strategy_wealth - market_wealth,
            "exposure_share": mean(row["position"] for row in rows),
            "position_changes": switches, "sessions": len(rows),
            "interval": paired_block_interval(paired_returns), "quarterly": quarterly,
            "negative_periods": sum(row["excess"] < 0 for row in quarterly)}


def build_signal_baseline(ihsg: dict[str, Any]) -> dict[str, Any]:
    """Require verified, unique, ordered, positive index closes; fail visibly."""
    blocked = {"ok": False, "schema_version": VERSION, "status": "INSUFFICIENT_EVIDENCE", "cases": []}
    if not ihsg.get("ok") or ihsg.get("data_quality") != "VERIFIED":
        return {**blocked, "reason": "Snapshot IHSG belum lolos pemeriksaan provenance."}
    try:
        series = [{"date": row["date"], "price": float(row["price"])} for row in ihsg["series"]]
        for row in series:
            date.fromisoformat(row["date"])
            if not math.isfinite(row["price"]) or row["price"] <= 0:
                raise ValueError("invalid price")
        dates = [row["date"] for row in series]
        if dates != sorted(set(dates)):
            raise ValueError("dates must be unique and ordered")
    except (KeyError, ValueError, TypeError, OverflowError):
        return {**blocked, "reason": "Tanggal/harga IHSG tidak valid, berulang, atau tidak berurutan."}
    if len(series) < 301:
        return {**blocked, "reason": "Perlu 200 sesi pemanasan, satu sesi jeda, dan setidaknya 100 return evaluasi."}
    fingerprint = hashlib.sha256(json.dumps(series, separators=(",", ":"), sort_keys=True).encode()).hexdigest()
    return {"ok": True, "schema_version": VERSION, "status": "EXPLORATORY_HISTORICAL_REPLAY",
            "source_class": "derived_metric", "provider": "Sectors.app",
            "source_fingerprint": fingerprint, "source_files": ihsg.get("source_files", []),
            "source_retrieved_at": ihsg.get("retrieved_at"), "observation_count": len(series),
            "source_coverage_start": dates[0], "source_coverage_end": dates[-1],
            "source_latest_price": series[-1]["price"], "warmup_sessions": 200,
            "rule": "IHSG close > SMA200 and SMA50 > SMA200; otherwise cash",
            "signal_lag_sessions": 2, "execution_delay_sessions": 1,
            "cash_return_annual": 0.0, "default_cost_bps": 10,
            "cases": [_replay(series, cost) for cost in (0, 10, 25)],
            "limits": ["Fixed-rule rolling historical replay; no fitted model or untouched holdout. Parameters were not searched.",
                       "IHSG is a price-index research proxy, not a directly tradable asset; actual fills, dividends and fund tracking are not modeled.",
                       "Costs of 0/10/25 bps per one-way exposure change are assumptions, not a broker fee schedule. Cash earns 0%; no terminal liquidation.",
                       "Paired-block interval conditions on the observed path; it does not replay a strategy on simulated markets or guarantee future returns."]}

## 11 · Perhitungan historis
Tidak ada training atau parameter search. Jalankan satu aturan tetap pada biaya 0/10/25 bps; 10 bps adalah default asumsi UI. Ini analisis proksi indeks; tidak ada terminal liquidation.

In [ ]:
result=build_signal_baseline(ihsg)
assert result['ok'], result.get('reason')
net=next(c for c in result['cases'] if c['cost_bps']==10)
print('Status:', result['status'])
print('Sesi:', net['sessions'], 'pergantian:', net['position_changes'], 'eksposur:', f"{net['exposure_share']:.1%}")

## 12 · Diagnostik, uncertainty dan failure cases
Tanggal aturan harus selalu mendahului tanggal pembentukan posisi dan return. Laporan kuartal menyimpan periode yang tertinggal. Interval bootstrap 95% adalah rentang estimasi selisih return aritmetika disetahunkan, conditional pada sampel ini.

In [ ]:
path=pd.DataFrame(net['path'])
for row in net['path'][1:]:
    assert row['signal_date'] < row['position_date'] < row['date']
assert math.isclose(math.prod(1+row['signal_return'] for row in net['path'][1:]), net['path'][-1]['signal_wealth'])
display(pd.DataFrame(net['quarterly']))
display(pd.DataFrame([net['interval']]))
print('Periode tertinggal:', net['negative_periods'])

## 13 · Hasil dan visual
Dua kurva memakai tanggal dan awal wealth identik. Grafik drawdown memperlihatkan kerugian dari puncak sebelumnya. Perhatikan hasil negatif; biaya dan exposure memengaruhi hasil.

In [ ]:
display(pd.DataFrame([{'cost_bps':c['cost_bps'],'signal_return':c['signal']['cumulative_return'],'market_return':c['market']['cumulative_return'],'excess_pp':c['excess_cumulative']*100,'signal_mdd':c['signal']['max_drawdown'],'market_mdd':c['market']['max_drawdown']} for c in result['cases']]))
fig,ax=plt.subplots(2,1,figsize=(12,7),sharex=True)
dates=pd.to_datetime(path.date)
for field,label,color in [('signal_wealth','Sinyal regime','#159b80'),('market_wealth','IHSG buy-and-hold','#397dc0')]:
    ax[0].plot(dates,(path[field]-1)*100,label=label,color=color)
for field,label,color in [('signal_drawdown','Sinyal regime','#159b80'),('market_drawdown','IHSG','#397dc0')]:
    ax[1].plot(dates,path[field]*100,label=label,color=color)
ax[0].set(title='Signal vs Market Baseline · biaya 10 bps',ylabel='Return kumulatif (%)')
ax[1].set(ylabel='Drawdown (%)',xlabel='Tanggal')
for a in ax: a.grid(alpha=.2); a.axhline(0,color='gray',linewidth=.7); a.legend()
plt.tight_layout();plt.show()

## 14 · Keputusan dan batas replay
Ini diagnostik fixed-rule price index. Tidak ada order, nominal modal, eksekusi broker atau nasihat akun. Biaya/slippage adalah asumsi sensitivitas; tracking error dan capacity produk investasi belum diukur.

In [ ]:
print('Cash yield: 0%; biaya satu arah 0/10/25 bps; tidak ada likuidasi akhir.')

## 15 · Kesimpulan dan keterbatasan
Hasil pada sampel ini dapat positif atau negatif. Membuktikan keunggulan memerlukan observasi baru, fill instrumen yang dapat diperdagangkan, biaya aktual, dan uji stabilitas yang disepakati. Interval tidak menjadi forecast.

In [ ]:
print('Selisih terhadap IHSG:', f"{net['excess_cumulative']*100:+.2f} pp")
print('Tertinggal pada',net['negative_periods'],'dari',len(net['quarterly']),'periode')
for limitation in result['limits']: print('-',limitation)

## 16 · Manifest reproducibility
Data hash, versi formula, parameter dan tanggal menjaga reproduksibilitas. Notebook tetap belum dieksekusi saat diserahkan; semua angka tampil setelah pengguna menjalankannya.

In [ ]:
manifest={'formula_version':VERSION,'input_sha256':hashlib.sha256(INPUT.read_bytes()).hexdigest(),'source_fingerprint':result['source_fingerprint'],'retrieved_at':ihsg.get('retrieved_at'),'source_files':ihsg['source_files'],'seed':42,'simulations':500,'block_sessions':20,'warmup_sessions':200,'execution_delay_sessions':1,'source_start':prices.date.iloc[0],'source_end':prices.date.iloc[-1],'status':result['status'],'python':platform.python_version()}
display(pd.DataFrame([manifest]))
artifact=bundle.get('signal_baseline')
if artifact is not None:
    assert artifact==result, 'Artefak web berbeda; regenerate dengan versi fungsi/data yang sama.'
    print('Hasil notebook cocok dengan artefak web.')